# 4장 파이썬 기초 II — 함수, 모듈, 파일, 예외, 클래스

**2단계 · 주석 달기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch04_2_comment.ipynb)

이 노트북에는 **주석을 지운 코드**가 들어 있습니다. 아래 표의 「주석 달기」 과제대로 내 말로 주석을 달아 보세요. 주석을 다는 동안 코드를 한 줄씩 다시 읽게 됩니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 셀을 실행하면 내가 고친 내용이 파일에 저장됩니다.
- 그 아래 확인 셀을 실행해, 주석을 단 뒤에도 코드가 전과 똑같이 도는지 봅니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01`~`p05`, `caselib.py`, `case_report.py` 를 한 줄씩 치고 출력 대조. `case_report.py` 는 인자 없이 한 번, `cases_all.csv` 를 열어 행 하나를 고친 뒤 다시 한 번 | [ch04_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch04_1_follow.ipynb) |
| 2. 주석 달기 | `case_report.py` 의 함수마다 "받는 것 / 돌려주는 것 / 실패하면" 세 줄 주석을 위에 달기 | [ch04_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch04_2_comment.ipynb) **(지금 이 노트북)** |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `case_report.py` 복원. 그다음 `Case` 에 `year()` 메서드를 추가하고 `report` 에 "연도별 건수"를 넣기 | [ch04_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch04_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "04"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/04")

## `case_report.py` — 대상: 파일 전체

In [ ]:
%%writefile case_report.py
import csv
import sys

from caselib import count_by


class Case:
    def __init__(self, case_no, case_type, claim, result, days):
        self.case_no = case_no
        self.type = case_type
        self.claim = claim
        self.result = result
        self.days = days

    @classmethod
    def from_row(cls, row):
        """CSV 한 행(딕셔너리, 값은 전부 글자)에서 Case를 만든다. 형식이 틀리면 ValueError."""
        return cls(
            row["case_no"],
            row["type"],
            int(row["claim"].replace(",", "")),
            row["result"],
            int(row["days"]),
        )

    def __repr__(self):
        return f"Case({self.case_no}, {self.type}, {self.claim:,}, {self.result}, {self.days}일)"


SAMPLE = [
    ["case_no", "type", "claim", "result", "days"],
    ["2099가합1001", "손해배상", "420000000", "인용", "310"],
    ["2099가합1002", "대여금", "150,000,000", "기각", "198"],
    ["2099가합1003", "손해배상", "880000000", "일부인용", "455"],
    ["2099가합1004", "부당이득", "67000000", "인용", "120"],
    ["2099가합1005", "손해배상", "230000000", "기각", "260"],
    ["2099가합1006", "손해배상", "미정", "인용", "90"],
]


def ensure_sample(path):
    """예제 파일이 없으면 만든다."""
    try:
        open(path, encoding="utf-8-sig").close()
    except FileNotFoundError:
        with open(path, "w", encoding="utf-8-sig", newline="") as f:
            csv.writer(f).writerows(SAMPLE)
        print(f"예제 파일을 만들었습니다: {path}")


def load_cases(path):
    cases, skipped = [], []
    with open(path, encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            try:
                cases.append(Case.from_row(row))
            except (ValueError, KeyError) as e:
                skipped.append((row.get("case_no", "?"), str(e)))
    return cases, skipped


def report(cases):
    records = [c.__dict__ for c in cases]
    print(f"사건 {len(cases)}건, 청구금액 합계 {sum(c.claim for c in cases):,}원")
    print("유형별 건수:", count_by(records, "type"))
    print("결과별 건수:", count_by(records, "result"))
    longest = max(cases, key=lambda c: c.days)
    print(f"최장 처리: {longest.case_no} ({longest.days}일)")


if __name__ == "__main__":
    path = sys.argv[1] if len(sys.argv) > 1 else "cases_all.csv"
    ensure_sample(path)
    cases, skipped = load_cases(path)
    report(cases)
    for case_no, why in skipped:
        print(f"건너뜀 {case_no}: {why}")

### 확인

In [ ]:
!python case_report.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- case_report.py